<a href="https://colab.research.google.com/github/zelal-Eizaldeen/deeplearning_course/blob/main/8_1demo_Llama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Llama Model

- In this demo, we will look a little bit about how to using the pre-trained **Llama model**

Llama is a **openly released model from Meta**.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Llama 3 Download and Run on Google Colab
- Prerequisites: Get Your Hugging Face Token
To access Llama 3 models, you must be logged into Hugging Face and have accepted Meta's license agreement.

1. Accept License: Go to the official Llama 3 Hugging Face page and accept the terms.

2. Get Token: Go to your Hugging Face Settings > Access Tokens and generate a Read token.


# Set Up the Colab Environment
- Start a new Colab notebook and set the hardware accelerator.

**Click Runtime > Change runtime type.**

- Set Hardware accelerator to GPU (T4 is preferred).

- Execute the following cells to install required libraries and log in to Hugging Face.

In [2]:
# 1. Install necessary libraries
!pip install -q torch transformers accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 16.2 MB/s eta 0:00:00


In [3]:
# 2. Log in to Hugging Face
# You will be prompted to enter your Hugging Face Access Token.
from huggingface_hub import login
login() # Enter your token when prompted

# Configure and Download the Model
We will use 4-bit quantization via the bitsandbytes library. This is crucial for fitting the model into Colab's free T4 GPU.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Define the model ID
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# 2. Configure 4-bit (QLoRA) quantization
# This is necessary to fit the model into Colab's VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # Use bfloat16 for computation on T4
)

# 3. Load the model and tokenizer
# map_location='auto' ensures it loads onto the GPU
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" # Distributes model layers across available devices (the GPU)
)

print("Llama 3 8B Instruct model downloaded and loaded successfully.")

#Generate Text (Inference)
Now you can use the loaded model for text generation.

Llama 3 uses a specific chat template (<|begin_of_text|>...) that the tokenizer handles automatically.

In [5]:
# 1. Define the input prompt
prompt = "Explain why the sky is blue in a concise and friendly manner."

# 2. Tokenize the prompt
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 3. Generate the response
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200, # Max length of the generated response
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id # Prevents warnings
    )

# 4. Decode and print the result
response = tokenizer.decode(outputs[0], skip_special_tokens=False)
# Optional: Find and clean the actual response text
# The response includes the original prompt and model tags
clean_response = response.split(prompt)[-1].strip()

print("\n--- Model Response ---")
print(clean_response)


--- Model Response ---
It's a great way to start the day with a bit of science!
The sky appears blue because of a phenomenon called Rayleigh scattering. When sunlight enters Earth’s atmosphere, it encounters tiny molecules of gases like nitrogen and oxygen. These molecules scatter the light in all directions, but they scatter shorter (blue) wavelengths more than longer (red) wavelengths. This is why the sky often appears blue during the daytime.
The scattering effect is strongest when the sun is high in the sky, which is why the sky tends to appear more vibrant during these times. Additionally, the amount of scattering also depends on the amount of particles in the atmosphere, which is why the sky can appear more hazy or polluted on certain days.
So, the next time you gaze up at a bright blue sky, remember that it's not just a pretty sight – it's also a result of some amazing science! #blueSky #RayleighScattering #ScienceForBeginners
The post Why is the Sky Blue?


 you should then be able to use this model for inference, so we are not training a model here, we have a pre-trained model.